OSU AI 535, Winter 2026
Group: Jacob Beitler, Jonathan Hotchkiss, Seth Mackovjak

Overview:
1) Data preprocessing: Import, structure, split, and augment the data.
2) Model definition: ResNetRS50 binary categorization.
3) Training loop definition: Incorporate appropriate Weights and Bias.
3) Model saving.

In [ ]:
# Import ALL_IDB2 dataset and build a binary ResNet50 model
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

from dataset_builder import idb2_dataset, category_balance_plot

# Image preprocessing (ResNet50 expects 224x224 RGB input normalized like ImageNet)
transform = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.Resize((224, 224)),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
        # mean=ResNet50_Weights.IMAGENET1K_V2.transforms.mean,
        # std=ResNet50_Weights.IMAGENET1K_V2.transforms.std
        ),
])

# Create train/test Datasets with transforms applied
train_dataset = idb2_dataset(train=True, transform=transform)
test_dataset = idb2_dataset(train=False, transform=transform)

category_balance_plot(train_dataset=train_dataset,
                      test_dataset=test_dataset)

# Build a binary ResNet50 model (output logits for a single class)
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)

# Use BCEWithLogitsLoss for binary classification
criterion = nn.BCEWithLogitsLoss()

# Configure device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model built:", model)
print("Device:", device)


In [ ]:
# Helper to build ResNet models with different depths for experimentation
from torchvision.models import (
    resnet18, resnet34, resnet50, resnet101, resnet152,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    ResNet101_Weights, ResNet152_Weights,
)

_resnet_builders = {
    "resnet18": (resnet18, ResNet18_Weights),
    "resnet34": (resnet34, ResNet34_Weights),
    "resnet50": (resnet50, ResNet50_Weights),
    "resnet101": (resnet101, ResNet101_Weights),
    "resnet152": (resnet152, ResNet152_Weights),
}


def build_resnet_binary(model_name: str = "resnet50", pretrained: bool = True, device=None) -> nn.Module:
    """Build a binary classification ResNet model.

    Args:
        model_name: One of resnet18/34/50/101/152.
        pretrained: If True, loads Imagenet weights.
        device: torch device (e.g. torch.device("cuda") or torch.device("cpu")).

    Returns:
        A torch.nn.Module configured for binary output (1 logit).
    """

    if model_name not in _resnet_builders:
        raise ValueError(f"Unsupported model: {model_name}. Choose from {list(_resnet_builders)}")

    builder, weights_cls = _resnet_builders[model_name]
    # IMAGENET1K_V2 wasn't availiable. Likely due to running torch 2.10...
    weights = weights_cls.IMAGENET1K_V1 if pretrained else None
    model = builder(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)

    if device is not None:
        model = model.to(device)

    return model


# Example: build multiple models for comparison
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Already defined above

models_to_try = ["resnet18", "resnet34", "resnet50", "resnet101", "resnet152"]
built_models = {name: build_resnet_binary(name, pretrained=True, device=device) for name in models_to_try}

print("Built models:", list(built_models.keys()))
print("Example model summary (resnet50):", built_models["resnet50"])


### Cell 4 Overview: Train and Test Loop
This cell takes the existing `model`, `train_dataset`, `test_dataset`, `criterion`, and `device` from earlier cells, then trains and evaluates using the hyperparameters in `hparams`. The metrics used are: loss, accuracy, error rate, F1 score, and confusion scores.

This cell does the following:
- Defines hyperparameters in `hparams`.
- Starts Weights & Biases logging.
- Uses manual batching without a dataloader.
- Trains the model each epoch and evaluates on test sets.
- Logs per-iteration and per-epoch metrics.
- Applies early stopping when the monitored metric plateaus.
- Saves the best checkpoint to the path in `hparams["model_save_path"]`.

How to use it:
1. Run previous cells to build datasets/model/loss/device.
2. Edit `hparams` for particular tests.
3. Run cell and and watch live updates through W&B.

In [ ]:
# Manual batching loop w/o dataloader + Weights & Biases logging
import time
import wandb

# Centralized hyperparameters and run config
# Moved to the train_model() function.

# Start W&B run for experiment tracking.
wandb.login()


def dataset_batch_iterator(dataset, batch_size, shuffle=False):
    """Yield mini-batches directly from a Dataset without DataLoader."""
    num_samples = len(dataset)

    # Shuffle for training, ordered pass for evaluation.
    if shuffle:
        indices = torch.randperm(num_samples).tolist()
    else:
        indices = list(range(num_samples))

    # Slice index list into mini-batches.
    for start in range(0, num_samples, batch_size):
        batch_indices = indices[start:start + batch_size]

        # Collect images/labels and stack into tensors.
        images = []
        labels = []
        for idx in batch_indices:
            image, label = dataset[idx]
            images.append(image)
            labels.append(label)

        images = torch.stack(images)
        labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

        yield images, labels


def train_one_epoch(model, dataset, criterion, optimizer, device, batch_size,
                    threshold, epoch, global_step, kwargs:dict):
    """Train for one epoch and return average train metrics."""
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    step_in_epoch = 0

    max_batches_per_epoch = kwargs.get('max_batches_per_epoch')
    max_total_steps = kwargs.get('max_total_steps')
    log_every_n_steps = kwargs.get('log_every_n_steps')

    for images, labels in dataset_batch_iterator(dataset, batch_size=batch_size, shuffle=True):
        # Optional iteration limits for faster debugging/controlled runs.
        if max_batches_per_epoch is not None and step_in_epoch >= max_batches_per_epoch:
            break

        if max_total_steps is not None and global_step >= max_total_steps:
            break

        # Move data to CPU/GPU.
        images = images.to(device)
        labels = labels.to(device)

        # Standard training step: forward -> loss -> backward -> optimizer step.
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        # Convert logits to probabilities, then threshold to class predictions.
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        batch_n = labels.size(0)
        batch_acc = (preds == labels).float().mean().item()

        total_loss += loss.item() * batch_n
        total_correct += (preds == labels).sum().item()
        total_samples += batch_n

        # Per-iteration logging to W&B.
        if global_step % log_every_n_steps == 0:
            wandb.log(
                {
                    "iter/loss": loss.item(),
                    "iter/accuracy": batch_acc,
                    "iter/epoch": epoch,
                    "iter/step_in_epoch": step_in_epoch,
                    "iter/global_step": global_step,
                },
                step=global_step,
            )

        step_in_epoch += 1
        global_step += 1

    avg_loss = total_loss / max(1, total_samples)
    avg_acc = total_correct / max(1, total_samples)
    return avg_loss, avg_acc, global_step, step_in_epoch


@torch.no_grad()
def evaluate(model, dataset, criterion, device, batch_size, threshold):
    """Evaluate on the test set and return loss/accuracy/error and confusion stats."""
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    tp = 0
    tn = 0
    fp = 0
    fn = 0

    for images, labels in dataset_batch_iterator(dataset, batch_size=batch_size, shuffle=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        batch_n = labels.size(0)
        total_loss += loss.item() * batch_n
        total_correct += (preds == labels).sum().item()
        total_samples += batch_n

        # Track confusion matrix values for more detailed metrics.
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    avg_loss = total_loss / max(1, total_samples)
    avg_acc = total_correct / max(1, total_samples)
    error_rate = 1.0 - avg_acc

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    metrics = {
        "loss": avg_loss,
        "accuracy": avg_acc,
        "error_rate": error_rate,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }
    return metrics


def is_improved(current_value, best_value, mode, min_delta):
    """Return True if monitored metric improved enough to reset early stopping."""
    if best_value is None:
        return True
    if mode == "min":
        return (best_value - current_value) > min_delta
    if mode == "max":
        return (current_value - best_value) > min_delta
    raise ValueError(f"Unsupported early stopping mode: {mode}")

In [ ]:
# Compile test combinations:
# model, optimizer, batch_size

from itertools import product

architectures = models_to_try.copy()
optimizers = ['adam', 'adamw', 'sgd']
learning_rates = [5e-5, 1e-4, 5e-4, 1e-3]
weight_decays = [5e-5, 1e-4, 5e-4]
batch_sizes = [8, 16, 32, 64]

tuning_tests = list(product(architectures, optimizers, learning_rates, weight_decays, batch_sizes))

print(f'The following {len(tuning_tests)} combinations will be tested:')
print(f'{'architecture':<13}| {'optimizer':10}| {'lr':10}| {'weight decay':<13}| {'batch size':<15}')
for test in tuning_tests:
    print(f'{test[0]:13}| {test[1]:10}| {test[2]:1.5f}   | {test[3]:<1.5f}      | {test[4]:<2d}')

In [ ]:
# Training function:
from copy import deepcopy

def train_model(architecture, optimizer, lr=1e-4, weight_decay=1e-4, batch_size=32, epochs=25):
    
    # Centralized hyperparameters and run config
    # Edit this dictionary to change training behavior.

    hparams = {
        "project": f"ai535-{architecture}-binary",
        "run_name": f"{architecture}-all_idb2-binary",
        "architecture": architecture,
        "dataset": "ALL_IDB2",
        "task": "binary-classification",
        "pretrained": True,
        "loss": "BCEWithLogitsLoss",
        "optimizer": optimizer,            # options: adam, adamw, sgd
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "epochs": epochs,
        "decision_threshold": 0.5,
        "max_batches_per_epoch": None,     # set int to cap iterations per epoch
        "max_total_steps": None,           # set int to cap total optimizer steps
        "log_every_n_steps": 5,
        "early_stopping_metric": "test/loss",
        "early_stopping_mode": "min",    # min for loss, max for accuracy
        "early_stopping_patience": 4,
        "early_stopping_min_delta": 1e-4,
        "model_save_path": f"./models/{architecture}_binary_all_idb2.pt",
    }
    
    run = wandb.init(
        project=hparams["project"],
        name=hparams["run_name"],
        config=hparams,
    )

    # Pulling the model from the prebuilt models:
    model = deepcopy(built_models[architecture])
    model.to(device)

    # Track model graph, gradients, and parameter updates.
    wandb.watch(model, criterion, log="all", log_freq=max(1, hparams["log_every_n_steps"]))

    # Pull frequently used values into local variables.
    batch_size = hparams["batch_size"]
    num_epochs = hparams["epochs"]
    learning_rate = hparams["learning_rate"]
    weight_decay = hparams["weight_decay"]
    threshold = hparams["decision_threshold"]
    max_batches_per_epoch = hparams["max_batches_per_epoch"]
    max_total_steps = hparams["max_total_steps"]
    log_every_n_steps = max(1, hparams["log_every_n_steps"])

    # Build the optimizer selected in hparams.
    optimizer_name = hparams["optimizer"].lower()
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {hparams['optimizer']}")

    # Early stopping / best-checkpoint tracking state.
    best_metric = None
    best_epoch = 0
    epochs_without_improvement = 0
    global_step = 0
    training_start = time.time()

    # Main training loop.
    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()

        # Train one full epoch.
        train_loss, train_acc, global_step, batches_seen = train_one_epoch(
            model=model,
            dataset=train_dataset,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            batch_size=batch_size,
            threshold=threshold,
            epoch=epoch,
            global_step=global_step,
            kwargs=hparams
        )

        # Evaluate on test set at the end of the epoch.
        test_metrics = evaluate(
            model=model,
            dataset=test_dataset,
            criterion=criterion,
            device=device,
            batch_size=batch_size,
            threshold=threshold,
        )

        epoch_seconds = time.time() - epoch_start

        print(
            f"Epoch [{epoch}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_metrics['loss']:.4f} | Test Acc: {test_metrics['accuracy']:.4f} | "
            f"Test Err: {test_metrics['error_rate']:.4f} | Batches: {batches_seen}"
        )

        # Per-epoch logging (train + test) to W&B.
        wandb.log(
            {
                "epoch": epoch,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "train/error_rate": 1.0 - train_acc,
                "train/batches_seen": batches_seen,
                "test/loss": test_metrics["loss"],
                "test/accuracy": test_metrics["accuracy"],
                "test/error_rate": test_metrics["error_rate"],
                "test/precision": test_metrics["precision"],
                "test/recall": test_metrics["recall"],
                "test/f1": test_metrics["f1"],
                "test/tp": test_metrics["tp"],
                "test/tn": test_metrics["tn"],
                "test/fp": test_metrics["fp"],
                "test/fn": test_metrics["fn"],
                "runtime/epoch_seconds": epoch_seconds,
                "runtime/global_step": global_step,
            },
            step=global_step,
        )

        # Select monitored metric for early stopping.
        monitor_key = hparams["early_stopping_metric"]
        if monitor_key.startswith("test/"):
            monitored_value = test_metrics[monitor_key.split("/", 1)[1]]
        elif monitor_key.startswith("train/"):
            monitored_value = train_loss if monitor_key.endswith("loss") else train_acc
        else:
            raise ValueError(f"Unsupported early_stopping_metric: {monitor_key}")

        # Save checkpoint when metric improves.
        if is_improved(monitored_value, best_metric, hparams["early_stopping_mode"], hparams["early_stopping_min_delta"]):
            best_metric = monitored_value
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), hparams["model_save_path"])
        else:
            epochs_without_improvement += 1

        # Stop when metric plateaus for epochs.
        if epochs_without_improvement >= hparams["early_stopping_patience"]:
            print(
                f"Early stopping at epoch {epoch}: "
                f"{monitor_key} plateaued for {hparams['early_stopping_patience']} epoch(s)."
            )
            break

        # Stop if hard step limit is reached.
        if max_total_steps is not None and global_step >= max_total_steps:
            print(f"Stopping: reached max_total_steps={max_total_steps}")
            break

    training_seconds = time.time() - training_start

    # Final evaluation after training stops.
    final_test_metrics = evaluate(
        model=model,
        dataset=test_dataset,
        criterion=criterion,
        device=device,
        batch_size=batch_size,
        threshold=threshold,
    )

    print("\nFinal Test Metrics:")
    for k, v in final_test_metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

    # Save final run summary fields in W&B.
    wandb.summary["best/epoch"] = best_epoch
    wandb.summary["best/metric_name"] = hparams["early_stopping_metric"]
    wandb.summary["best/metric_value"] = best_metric if best_metric is not None else float("nan")
    wandb.summary["runtime/training_seconds"] = training_seconds

    wandb.summary["final_test/loss"] = final_test_metrics["loss"]
    wandb.summary["final_test/accuracy"] = final_test_metrics["accuracy"]
    wandb.summary["final_test/error_rate"] = final_test_metrics["error_rate"]
    wandb.summary["final_test/precision"] = final_test_metrics["precision"]
    wandb.summary["final_test/recall"] = final_test_metrics["recall"]
    wandb.summary["final_test/f1"] = final_test_metrics["f1"]
    wandb.summary["final_test/tp"] = final_test_metrics["tp"]
    wandb.summary["final_test/tn"] = final_test_metrics["tn"]
    wandb.summary["final_test/fp"] = final_test_metrics["fp"]
    wandb.summary["final_test/fn"] = final_test_metrics["fn"]

    print(f"\nBest checkpoint saved to {hparams['model_save_path']}")
    wandb.save(hparams["model_save_path"])
    wandb.finish()

In [ ]:
# Train a variety of combinations:

for test in tuning_tests:
    train_model(*test)